In [2]:
import pandas as pd
import re

# Last opp filen manuelt via Colab GUI og bruk riktig filnavn her
df = pd.read_csv("lifeprintscrape.csv")

# Gi kolonnen et enklere navn
df.columns = ['ASL']

# Funksjon for å filtrere ut "støy"
def is_noise(ASL):
    ASL = str(ASL).strip()

    # Fjern tomme rader
    if not ASL:
        return True

    # Fjern .mp4 og lignende filtyper
    if re.search(r'\.(mp4|mov|avi|wmv)$', ASL.lower()):
        return True

    # Fjern metadata og undervisnings-relaterte nøkkelord
    if re.search(r'\b(lesson|unit|practice|test|quiz|review|drill|part \d+|version|file|activity)\b', ASL.lower()):
        return True

    # Fjern rader med bare ett ord eller et isolert tegn/tall
    if len(ASL.split()) == 1:
        return True
    if ASL.strip().isdigit():
        return True

    # Fjern klokkeslett eller tidspunkter
    if re.search(r'^\d{1,2}:\d{2}( ?[ap]m)?$', ASL.lower()):
        return True

    # Fjern flytende engelsk (stor forbokstav og apostrof eller vanlig spørresetning)
    if ASL[0].isupper() and ("'" in ASL or ASL.endswith("?")):
        return True

    # Fjern setninger som er mest små bokstaver (ser ut som engelsk)
    words = ASL.split()
    lowercase_count = sum(1 for word in words if word.islower())
    if len(words) > 0 and (lowercase_count / len(words)) > 0.5:
        return True

    return False

# Bruk funksjonen på datasettet
df["is_noise"] = df["ASL"].apply(is_noise)

# Del opp i beholdt og fjernet
clean_df = df[~df["is_noise"]].copy()
removed_df = df[df["is_noise"]].copy()

# Lagre resultatene (valgfritt)
clean_df.to_csv("lifeprint_cleaned.csv", index=False)
removed_df.to_csv("lifeprint_removed.csv", index=False)

# Vis hvor mye som ble beholdt og fjernet
print(f"Beholdt: {len(clean_df)} rader")
print(f"Fjernet: {len(removed_df)} rader")


Beholdt: 1754 rader
Fjernet: 10456 rader


In [10]:
# Åpne filen som ren tekst
with open("lifeprint_cleaned_reformatted.csv", "r", encoding="utf-8") as f:
    lines = f.readlines()

# Hent første kolonne fra hver rad (før første komma)
asl_only = [line.split(",")[0].strip() for line in lines if line.strip()]

# Fjern eventuell header
if asl_only[0].lower().startswith("asl"):
    asl_only = asl_only[1:]

# Lag en DataFrame
import pandas as pd
df_asl = pd.DataFrame(asl_only, columns=["ASL"])

# Lagre som ny fil
df_asl.to_csv("ASL_LIFEPRINT_NEW.csv", index=False)

# Vis noen rader
df_asl.head()

,ASL
0,Help you;
1,#BACK
2,$120 MINUS 98 CENTS EQUAL HOW-MUCH? *[$119.02];
3,[sweep]-INDEX
4,[The] HE-(point off to your side) BOY LIKE[s] ...


In [11]:
import pandas as pd

# Les inn forrige fil
df = pd.read_csv("/content/ASL_LIFEPRINT_NEW.csv")

# Fjern semikolon på slutten av hver rad (hvis det finnes)
df['ASL'] = df['ASL'].str.replace(r';\s*$', '', regex=True)

# Lagre til ny fil
df.to_csv("ASL_LIFEPRINT_CLEAN.csv", index=False)

# Vis noen rader
df.head()


,ASL
0,Help you
1,#BACK
2,$120 MINUS 98 CENTS EQUAL HOW-MUCH? *[$119.02]
3,[sweep]-INDEX
4,[The] HE-(point off to your side) BOY LIKE[s] ...
